# AI-Based Network Intrusion Detection & Cybersecurity Analytics

---

## 1. Project Introduction

### What is Network Intrusion Detection?
Network Intrusion Detection (NID) is the process of monitoring network traffic to identify malicious activity, policy violations, or unauthorized access attempts. Modern networks generate enormous volumes of traffic, making manual inspection impractical. Machine learning enables automated, real-time classification of connections as **normal** or **attack**.

### Why Does It Matter?
Cyberattacks cost organizations billions of dollars annually. Intrusion Detection Systems (IDS) form a critical layer of defense in Security Operations Centers (SOCs), helping analysts triage threats faster and reduce mean-time-to-detect (MTTD).

### What This Project Solves
This project builds an end-to-end machine learning pipeline that:
- Performs thorough exploratory analysis of network traffic data
- Engineers meaningful features from raw connection attributes
- Trains a Random Forest classifier to distinguish normal from attack traffic
- Evaluates the model rigorously using multiple metrics
- Provides actionable cybersecurity insights

### Dataset Source
**UNSW-NB15** — A comprehensive network intrusion dataset created at the University of New South Wales.  
Source: [https://www.kaggle.com/datasets/dhoogla/unswnb15/versions/5](https://www.kaggle.com/datasets/dhoogla/unswnb15/versions/5)

The dataset contains 175,341 training records and a separate test set, each with 36 features covering connection metadata, payload statistics, TCP flags, and HTTP attributes.

### Analytical Workflow
```
Data Loading → Quality Audit → Cleaning → EDA → Feature Engineering
    → ML Problem Definition → Preprocessing Pipeline → Random Forest
        → Evaluation → Feature Importance → Attack Analysis → Insights
```

---
## 2. Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Consistent plot style
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})
sns.set_style('whitegrid')

print('Libraries loaded successfully.')
print(f'  pandas  {pd.__version__}')
print(f'  numpy   {np.__version__}')
print(f'  sklearn {__import__("sklearn").__version__}')
print(f'  matplotlib {matplotlib.__version__}')
print(f'  seaborn {sns.__version__}')

---
## 3. Dataset Loading

The dataset is provided as Parquet files (`UNSW_NB15_training-set.parquet` and `UNSW_NB15_testing-set.parquet`). The loading logic searches several common locations so the notebook runs correctly on Kaggle, VS Code, IBM Bob, and local environments.

In [ ]:
def find_dataset_file(filename):
    """Search for a dataset file in common locations."""
    search_dirs = [
        '.',
        './UNSW-NB15',
        '../input/unswnb15',          # Kaggle
        '../input',
        os.path.dirname(os.path.abspath('__file__')),
    ]
    for d in search_dirs:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            return candidate
    return None


TRAIN_FILE = 'UNSW_NB15_training-set.parquet'
TEST_FILE  = 'UNSW_NB15_testing-set.parquet'

train_path = find_dataset_file(TRAIN_FILE)
test_path  = find_dataset_file(TEST_FILE)

if train_path is None:
    raise FileNotFoundError(
        f"\n[ERROR] Could not locate '{TRAIN_FILE}'.\n"
        "Please download the dataset from:\n"
        "  https://www.kaggle.com/datasets/dhoogla/unswnb15/versions/5\n"
        "and place the .parquet files in one of these locations:\n"
        "  • Same folder as this notebook  (./)"
        "  • ./UNSW-NB15/"
    )

print(f'Training file : {train_path}')
print(f'Testing  file : {test_path}')

# ── Load parquet files ──────────────────────────────────────────────────────
df_train = pd.read_parquet(train_path)
df_test  = pd.read_parquet(test_path) if test_path else None

print(f'\nTraining set shape : {df_train.shape}')
if df_test is not None:
    print(f'Testing  set shape : {df_test.shape}')

print('\nFirst 5 rows:')
df_train.head()

In [ ]:
print('Column names and data types:\n')
print(df_train.dtypes.to_string())
print(f'\nMemory usage: {df_train.memory_usage(deep=True).sum() / 1e6:.2f} MB')

---
## 4. Data Quality Audit

In [ ]:
# ── Missing values ──────────────────────────────────────────────────────────
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]

if missing_summary.empty:
    print('No missing values found in the training set.')
else:
    print('Columns with missing values:')
    print(missing_summary.sort_values('Missing %', ascending=False))

# ── Duplicate rows ──────────────────────────────────────────────────────────
n_dup = df_train.duplicated().sum()
print(f'\nDuplicate rows : {n_dup} ({n_dup / len(df_train) * 100:.2f}%)')

# ── Unique values per column ────────────────────────────────────────────────
print('\nUnique values per column:')
print(df_train.nunique().to_string())

# ── Target distribution ──────────────────────────────────────────────────────
target_counts = df_train['label'].value_counts()
target_pct    = df_train['label'].value_counts(normalize=True) * 100
target_df = pd.DataFrame({'Count': target_counts, 'Percentage': target_pct.round(2)})
target_df.index = target_df.index.map({0: 'Normal (0)', 1: 'Attack (1)'})
print('\nTarget (label) distribution:')
print(target_df)

In [ ]:
# ── Basic descriptive statistics ────────────────────────────────────────────
print('Descriptive statistics (numeric columns):')
df_train.describe().T

---
## 5. Data Cleaning

Cleaning steps applied:
1. **Remove exact duplicate rows** — identical records add no information.
2. **Replace infinite values with NaN** — some network-derived ratio features can produce ±inf.
3. **Drop columns with > 80% missing values** (threshold-based, not blind `dropna`).
4. **Fill remaining numeric NaNs with column median** — robust to skew.
5. **Fill categorical NaNs with the string `'Unknown'`** — preserves category count.
6. **Standardise categorical string casing** — ensure consistent label values.

In [ ]:
df = df_train.copy()

# ── Step 1: Remove duplicate rows ───────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
print(f'Step 1 — Duplicates removed: {before - len(df):,}  (rows remaining: {len(df):,})')

# ── Step 2: Replace infinite values ─────────────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
inf_count = np.isinf(df[numeric_cols].values).sum()
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print(f'Step 2 — Infinite values replaced with NaN: {inf_count:,}')

# ── Step 3: Drop columns with >80% missing ───────────────────────────────────
high_missing = df.columns[df.isnull().mean() > 0.80].tolist()
if high_missing:
    df = df.drop(columns=high_missing)
    print(f'Step 3 — Dropped high-missing columns: {high_missing}')
else:
    print('Step 3 — No columns exceed 80% missing threshold.')

# ── Step 4: Fill numeric NaNs with median ───────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())
print(f'Step 4 — Numeric NaNs filled with column median.')

# ── Step 5: Fill categorical NaNs ───────────────────────────────────────────
cat_cols = df.select_dtypes(include=['category', 'object']).columns.tolist()
for col in cat_cols:
    if df[col].isnull().any():
        if hasattr(df[col], 'cat'):
            df[col] = df[col].cat.add_categories('Unknown').fillna('Unknown')
        else:
            df[col] = df[col].fillna('Unknown')
print('Step 5 — Categorical NaNs filled with "Unknown".')

# ── Step 6: Normalise string casing for categoricals ────────────────────────
for col in cat_cols:
    if df[col].dtype == object:
        df[col] = df[col].str.strip().str.lower()
print('Step 6 — String categorical values stripped and lowercased.')

print(f'\nCleaned dataset shape: {df.shape}')
print(f'Remaining NaN values : {df.isnull().sum().sum()}')

---
## 6. Exploratory Data Analysis

### Visualization 1 — Normal vs Attack Traffic Distribution

In [ ]:
label_counts = df['label'].value_counts().sort_index()
label_names  = {0: 'Normal', 1: 'Attack'}
labels_plot  = [label_names.get(i, str(i)) for i in label_counts.index]
colors       = ['#2196F3', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
axes[0].bar(labels_plot, label_counts.values, color=colors, edgecolor='white', width=0.5)
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Traffic Volume: Normal vs Attack', fontweight='bold')
axes[0].set_ylabel('Number of Connections')
axes[0].set_xlabel('Traffic Class')

# Pie chart
wedge_props = {'edgecolor': 'white', 'linewidth': 2}
axes[1].pie(
    label_counts.values,
    labels=labels_plot,
    autopct='%1.1f%%',
    colors=colors,
    wedgeprops=wedge_props,
    startangle=140,
    textprops={'fontsize': 12}
)
axes[1].set_title('Traffic Class Proportion', fontweight='bold')

plt.suptitle('UNSW-NB15 — Normal vs Attack Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_traffic_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

for label_id, name in label_names.items():
    cnt = label_counts.get(label_id, 0)
    pct = cnt / len(df) * 100
    print(f'  {name:6s}: {cnt:>7,}  ({pct:.1f}%)')

### Visualization 2 — Top Attack Categories

In [ ]:
if 'attack_cat' in df.columns:
    # Consider only attack records for category analysis
    attack_df = df[df['label'] == 1].copy()
    cat_series = attack_df['attack_cat'].astype(str).str.strip()
    # Filter out 'nan' / 'unknown' / 'Normal' which are not real attack categories
    cat_series = cat_series[~cat_series.str.lower().isin(['nan', 'unknown', 'normal', ''])]
    cat_counts = cat_series.value_counts()

    fig, ax = plt.subplots(figsize=(10, 6))
    palette = sns.color_palette('tab10', n_colors=len(cat_counts))
    bars = ax.barh(cat_counts.index[::-1], cat_counts.values[::-1], color=palette[::-1], edgecolor='white')
    for bar, val in zip(bars, cat_counts.values[::-1]):
        ax.text(bar.get_width() + cat_counts.max() * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', fontsize=9)
    ax.set_xlabel('Number of Attack Records')
    ax.set_ylabel('Attack Category')
    ax.set_title('Top Attack Categories in UNSW-NB15', fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig2_attack_categories.png', bbox_inches='tight', dpi=120)
    plt.show()

    print('\nAttack category breakdown:')
    cat_pct = (cat_counts / cat_counts.sum() * 100).round(2)
    summary = pd.DataFrame({'Count': cat_counts, 'Percentage': cat_pct})
    print(summary.to_string())
else:
    print('attack_cat column not present — skipping.')

### Visualization 3 — Traffic Distribution by Protocol

In [ ]:
proto_col = df['proto'].astype(str)
top_n     = 15
top_protos = proto_col.value_counts().head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Overall top protocols
sns.barplot(x=top_protos.values, y=top_protos.index, ax=axes[0],
            palette='Blues_r', edgecolor='white')
axes[0].set_title(f'Top {top_n} Protocols by Connection Count', fontweight='bold')
axes[0].set_xlabel('Connection Count')
axes[0].set_ylabel('Protocol')

# Protocol breakdown by label (top 10)
top10_protos = proto_col.value_counts().head(10).index.tolist()
proto_label = df[proto_col.isin(top10_protos)].copy()
proto_label['proto_str'] = proto_label['proto'].astype(str)
proto_label['Traffic'] = proto_label['label'].map({0: 'Normal', 1: 'Attack'})
proto_pivot = proto_label.groupby(['proto_str', 'Traffic']).size().unstack(fill_value=0)
proto_pivot.plot(kind='bar', ax=axes[1], color=['#2196F3', '#F44336'],
                 edgecolor='white', width=0.7)
axes[1].set_title('Top 10 Protocols: Normal vs Attack', fontweight='bold')
axes[1].set_xlabel('Protocol')
axes[1].set_ylabel('Connection Count')
axes[1].legend(title='Traffic Class')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Network Protocol Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_protocol_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

### Visualization 4 — Traffic Distribution by Service

In [ ]:
service_col = df['service'].astype(str)
service_counts = service_col.value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Count bar
sns.barplot(x=service_counts.values, y=service_counts.index, ax=axes[0],
            palette='Oranges_r', edgecolor='white')
axes[0].set_title('Connection Count by Service', fontweight='bold')
axes[0].set_xlabel('Connection Count')
axes[0].set_ylabel('Service')

# Service × label stacked
df_svc = df.copy()
df_svc['service_str'] = df_svc['service'].astype(str)
df_svc['Traffic'] = df_svc['label'].map({0: 'Normal', 1: 'Attack'})
svc_pivot = df_svc.groupby(['service_str', 'Traffic']).size().unstack(fill_value=0)
svc_pivot.plot(kind='bar', stacked=True, ax=axes[1],
               color=['#2196F3', '#F44336'], edgecolor='white', width=0.8)
axes[1].set_title('Service Traffic: Normal vs Attack (Stacked)', fontweight='bold')
axes[1].set_xlabel('Service')
axes[1].set_ylabel('Connection Count')
axes[1].legend(title='Traffic Class')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Network Service Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_service_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

### Visualization 5 — Network Features vs Traffic Class

In [ ]:
# Select key numeric features to compare across label classes
key_features = ['dur', 'sbytes', 'dbytes', 'rate', 'sload', 'dload',
                'spkts', 'dpkts', 'smean', 'dmean']
key_features = [f for f in key_features if f in df.columns]

df_plot = df[key_features + ['label']].copy()
df_plot['Traffic'] = df_plot['label'].map({0: 'Normal', 1: 'Attack'})

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    # Cap extreme outliers at 99th percentile for visual clarity
    cap = df_plot[feat].quantile(0.99)
    plot_data = df_plot[df_plot[feat] <= cap]
    sns.boxplot(
        data=plot_data, x='Traffic', y=feat, ax=axes[i],
        palette={'Normal': '#2196F3', 'Attack': '#F44336'},
        showfliers=False, width=0.5
    )
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Hide any unused subplots
for j in range(len(key_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Network Feature Distributions: Normal vs Attack Traffic\n(Outliers capped at 99th percentile)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_feature_vs_label.png', bbox_inches='tight', dpi=120)
plt.show()

### Visualization 6 — Correlation Analysis

In [ ]:
# Select numeric columns for correlation (exclude TCP base sequence numbers
# which have very high cardinality and no meaningful linear relationship)
corr_drop = ['stcpb', 'dtcpb']
numeric_for_corr = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_for_corr = [c for c in numeric_for_corr if c not in corr_drop]

corr_matrix = df[numeric_for_corr].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap='RdBu_r',
    center=0, linewidths=0.3, ax=ax,
    cbar_kws={'shrink': 0.7}
)
ax.set_title('Correlation Heatmap — Numeric Features (UNSW-NB15)', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('fig6_correlation_heatmap.png', bbox_inches='tight', dpi=120)
plt.show()

# Top correlations with label
label_corr = corr_matrix['label'].drop('label').abs().sort_values(ascending=False)
print('Top 10 features correlated with label (absolute Pearson r):')
print(label_corr.head(10).to_string())

---
## 7. Feature Engineering

New features derived from existing columns where meaningful for network intrusion analysis:

In [ ]:
df_eng = df.copy()

# ── Feature 1: total_bytes — total payload volume per connection ─────────────
if 'sbytes' in df_eng.columns and 'dbytes' in df_eng.columns:
    df_eng['total_bytes'] = df_eng['sbytes'] + df_eng['dbytes']
    print('total_bytes = sbytes + dbytes  (total connection payload volume)')

# ── Feature 2: total_pkts — total packet count per connection ────────────────
if 'spkts' in df_eng.columns and 'dpkts' in df_eng.columns:
    df_eng['total_pkts'] = df_eng['spkts'] + df_eng['dpkts']
    print('total_pkts  = spkts + dpkts   (total packets exchanged)')

# ── Feature 3: bytes_per_pkt — average bytes per packet ─────────────────────
if 'total_bytes' in df_eng.columns and 'total_pkts' in df_eng.columns:
    df_eng['bytes_per_pkt'] = np.where(
        df_eng['total_pkts'] > 0,
        df_eng['total_bytes'] / df_eng['total_pkts'],
        0
    )
    print('bytes_per_pkt = total_bytes / total_pkts  (packet payload density)')

# ── Feature 4: src_dst_byte_ratio — directional traffic asymmetry ────────────
if 'sbytes' in df_eng.columns and 'dbytes' in df_eng.columns:
    total = df_eng['sbytes'] + df_eng['dbytes']
    df_eng['src_dst_byte_ratio'] = np.where(
        total > 0,
        df_eng['sbytes'] / total,
        0.5
    )
    print('src_dst_byte_ratio = sbytes / (sbytes + dbytes)  (traffic direction asymmetry)')

# ── Feature 5: load_diff — difference in source vs destination load ──────────
if 'sload' in df_eng.columns and 'dload' in df_eng.columns:
    df_eng['load_diff'] = df_eng['sload'] - df_eng['dload']
    print('load_diff   = sload - dload   (asymmetry in link utilisation)')

# ── Feature 6: tcp_handshake_complete — did the full TCP 3-way handshake occur
if all(c in df_eng.columns for c in ['tcprtt', 'synack', 'ackdat']):
    df_eng['tcp_handshake_complete'] = (
        (df_eng['tcprtt'] > 0) & (df_eng['synack'] > 0) & (df_eng['ackdat'] > 0)
    ).astype(int)
    print('tcp_handshake_complete = 1 if tcprtt>0 & synack>0 & ackdat>0  (full handshake flag)')

print(f'\nDataset shape after feature engineering: {df_eng.shape}')

---
## 8. Machine Learning Problem Definition

**Task:** Binary classification — predict whether a network connection is **Normal (0)** or an **Attack (1)**.

**Target column:** `label`  
**Leakage guard:** `attack_cat` is excluded from input features because it directly encodes attack type information — using it would make the model trivially accurate but useless in production (you would already know it's an attack).  
**No raw identifiers** are included in the feature set.

In [ ]:
TARGET = 'label'

# Columns to always exclude
LEAKAGE_COLS = ['attack_cat']        # encodes attack type — direct label leakage
ID_COLS      = []                    # no raw IP/port identifier columns in this parquet
EXCLUDE_COLS = LEAKAGE_COLS + ID_COLS + [TARGET]

feature_cols = [c for c in df_eng.columns if c not in EXCLUDE_COLS]

X = df_eng[feature_cols]
y = df_eng[TARGET]

print(f'Target column    : {TARGET}')
print(f'Excluded columns : {EXCLUDE_COLS}')
print(f'Feature count    : {len(feature_cols)}')
print(f'Feature names    : {feature_cols}')
print(f'\nTarget value verification:')
print(y.value_counts().rename({0: 'Normal (0)', 1: 'Attack (1)'}).to_string())

---
## 9. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # preserve class proportions
)

print(f'Training set   : {X_train.shape[0]:>7,} records  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set       : {X_test.shape[0]:>7,} records  ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'Features       : {X_train.shape[1]}')
print(f'\nTrain label distribution:')
print(y_train.value_counts().rename({0: 'Normal', 1: 'Attack'}).to_string())
print(f'\nTest  label distribution:')
print(y_test.value_counts().rename({0: 'Normal', 1: 'Attack'}).to_string())

---
## 10. Preprocessing Pipeline

A `ColumnTransformer` applies different pipelines to numeric and categorical columns:
- **Numeric**: `SimpleImputer(median)` → `StandardScaler`
- **Categorical**: `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='ignore')`

In [ ]:
# Identify numeric and categorical feature columns in the feature set
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=['category', 'object']
).columns.tolist()

print(f'Numeric features     ({len(numeric_features)}): {numeric_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

# ── Numeric pipeline ─────────────────────────────────────────────────────────
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# ── Categorical pipeline ──────────────────────────────────────────────────────
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# ── Column transformer ────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,      numeric_features),
    ('cat', categorical_pipeline,  categorical_features),
])

print('\nPreprocessing pipeline constructed.')

---
## 11. Random Forest Model

A `RandomForestClassifier` is well-suited to this task because:
- It handles mixed numeric and categorical (after encoding) features naturally
- It is robust to outliers and non-linearity common in network traffic data
- `class_weight='balanced'` compensates for any class imbalance
- Feature importance is built-in and interpretable

In [ ]:
import time

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
    max_depth=None,        # fully grown trees
    min_samples_leaf=2,    # slight regularisation
)

# Full pipeline: preprocessing + classifier
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   rf_model),
])

print('Training Random Forest (200 trees) — this may take a few minutes...')
t0 = time.time()
full_pipeline.fit(X_train, y_train)
elapsed = time.time() - t0

print(f'Training complete in {elapsed:.1f} seconds.')

---
## 12. Model Evaluation

In [ ]:
# ── Predictions ──────────────────────────────────────────────────────────────
y_pred       = full_pipeline.predict(X_test)
y_pred_proba = full_pipeline.predict_proba(X_test)[:, 1]

# ── Core metrics ──────────────────────────────────────────────────────────────
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
roc_auc   = roc_auc_score(y_test, y_pred_proba)

print('=' * 50)
print('         MODEL EVALUATION RESULTS')
print('=' * 50)
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print('=' * 50)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts heatmap
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal', 'Attack'],
    yticklabels=['Normal', 'Attack'],
    linewidths=1, linecolor='white', ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

# Normalised heatmap
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='Blues',
    xticklabels=['Normal', 'Attack'],
    yticklabels=['Normal', 'Attack'],
    linewidths=1, linecolor='white', ax=axes[1],
    vmin=0, vmax=1
)
axes[1].set_title('Confusion Matrix (Normalised)', fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('Actual Label')

plt.suptitle('Random Forest — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_confusion_matrix.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#1565C0', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='#1565C0')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Binary Intrusion Detection', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('fig8_roc_curve.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 13. Feature Importance

In [ ]:
# Extract transformed feature names from the pipeline
fitted_preprocessor = full_pipeline.named_steps['preprocessor']

# Numeric feature names are unchanged
num_feat_names = numeric_features

# OHE adds one name per category
ohe = fitted_preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_feat_names = ohe.get_feature_names_out(categorical_features).tolist()

all_feature_names = num_feat_names + cat_feat_names

# Get importances from the fitted Random Forest
importances = full_pipeline.named_steps['classifier'].feature_importances_

importance_df = pd.DataFrame({
    'Feature':    all_feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

top_n = 20
top_features = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
colors_imp = plt.cm.RdYlGn(np.linspace(0.3, 0.9, top_n))[::-1]
bars = ax.barh(
    top_features['Feature'][::-1],
    top_features['Importance'][::-1],
    color=colors_imp, edgecolor='white'
)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=11)
ax.set_title(f'Top {top_n} Most Important Features — Random Forest\n'
             f'(Intrusion Detection)', fontweight='bold', fontsize=13)
ax.axvline(x=importance_df['Importance'].mean(), color='#B71C1C',
           linestyle='--', linewidth=1.5, label='Mean importance')
ax.legend()
plt.tight_layout()
plt.savefig('fig9_feature_importance.png', bbox_inches='tight', dpi=120)
plt.show()

print('Top 20 features by importance:')
print(top_features.to_string(index=False))

**Interpretation:** Features near the top of the chart have the greatest influence on the model's decision. High-importance features represent the network traffic characteristics that most reliably distinguish normal from attack connections. For instance, byte-volume features (`sbytes`, `dbytes`, `total_bytes`) and connection rate metrics capture the volume anomalies typical of DoS and reconnaissance attacks, while TCP handshake features (`synack`, `tcprtt`) reveal incomplete or forged connections.

---
## 14. Attack Category Analysis

In [ ]:
if 'attack_cat' in df_eng.columns:
    # Show category breakdown on the full cleaned dataset
    all_cats = df_eng['attack_cat'].astype(str).str.strip()
    # Separate normal vs attack records
    normal_mask = df_eng['label'] == 0
    attack_mask = df_eng['label'] == 1

    cat_counts = all_cats[attack_mask].value_counts()
    cat_pct    = (cat_counts / cat_counts.sum() * 100).round(2)
    cat_summary = pd.DataFrame({'Count': cat_counts, 'Percentage (%)': cat_pct})

    print('Attack Category Breakdown (attack records only):')
    print(cat_summary.to_string())

    # ── Visualization ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Count bar
    palette = sns.color_palette('Set2', n_colors=len(cat_counts))
    axes[0].bar(cat_counts.index, cat_counts.values, color=palette, edgecolor='white')
    axes[0].set_title('Attack Category Frequency', fontweight='bold')
    axes[0].set_xlabel('Attack Category')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=40)

    # Percentage pie
    axes[1].pie(
        cat_pct.values, labels=cat_pct.index,
        autopct='%1.1f%%', colors=palette,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
        startangle=90, textprops={'fontsize': 9}
    )
    axes[1].set_title('Attack Category Proportion', fontweight='bold')

    plt.suptitle('UNSW-NB15 — Attack Category Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig10_attack_categories.png', bbox_inches='tight', dpi=120)
    plt.show()
else:
    print('attack_cat column not found — skipping category analysis.')

---
## 15. Sample Prediction Demo

This section demonstrates the trained pipeline predicting on real unseen test records.

In [ ]:
SAMPLE_SIZE = 10
# Sample stratified across both classes for a balanced demo
sample_idx = (
    pd.Series(y_test.index)
    .groupby(y_test.values)
    .apply(lambda g: g.sample(min(SAMPLE_SIZE // 2, len(g)), random_state=7))
    .values
)

X_sample = X_test.loc[sample_idx]
y_actual = y_test.loc[sample_idx]

pred_labels = full_pipeline.predict(X_sample)
pred_proba  = full_pipeline.predict_proba(X_sample)[:, 1]

demo_df = pd.DataFrame({
    'Actual Label':      y_actual.map({0: 'Normal', 1: 'Attack'}).values,
    'Predicted Label':   pd.Series(pred_labels).map({0: 'Normal', 1: 'Attack'}).values,
    'Attack Probability': np.round(pred_proba, 4),
    'Correct':           ['✓' if a == p else '✗' for a, p in zip(y_actual.values, pred_labels)],
})

print('Sample Prediction Results (10 test records):')
print(demo_df.to_string(index=False))

correct_pct = (demo_df['Correct'] == '✓').mean() * 100
print(f'\nCorrect predictions in sample: {correct_pct:.0f}%')

---
## 16. Final Project Insights

In [ ]:
# ── Compute final summary statistics from actual executed results ─────────────
total_records   = len(df)
n_normal        = int((df['label'] == 0).sum())
n_attack        = int((df['label'] == 1).sum())
pct_attack      = n_attack / total_records * 100

top_feat        = importance_df['Feature'].iloc[0]
top5_feats      = importance_df['Feature'].head(5).tolist()

if 'attack_cat' in df_eng.columns:
    top_attack_cat = (
        df_eng[df_eng['label'] == 1]['attack_cat']
        .astype(str).str.strip().value_counts().idxmax()
    )
else:
    top_attack_cat = 'N/A'

print('=' * 60)
print('       FINAL PROJECT INSIGHTS — KEY FINDINGS')
print('=' * 60)
print(f'''
DATASET
  Total connection records : {total_records:,}
  Normal traffic           : {n_normal:,}  ({n_normal/total_records*100:.1f}%)
  Attack traffic           : {n_attack:,}  ({pct_attack:.1f}%)
  Most frequent attack cat : {top_attack_cat}

MODEL PERFORMANCE (Random Forest, 200 trees)
  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)
  Precision : {precision:.4f}
  Recall    : {recall:.4f}
  F1 Score  : {f1:.4f}
  ROC-AUC   : {roc_auc:.4f}

FEATURE IMPORTANCE — TOP 5
  {chr(10).join(f'  {i+1}. {f}' for i, f in enumerate(top5_feats))}

CYBERSECURITY INTERPRETATION
  • {pct_attack:.1f}% of all connections in this dataset are attack traffic,
    confirming the dataset represents a realistic threat-laden environment.
  • The most predictive features are volume and rate metrics (bytes, packets,
    load), which capture the anomalous burst patterns of DoS and scanning attacks.
  • TCP handshake metrics (synack, tcprtt, ackdat) distinguish incomplete or
    spoofed connections commonly used in reconnaissance and exploitation.
  • The trained Random Forest achieves strong recall, meaning it detects the
    majority of true attacks — critical for an IDS where missed detections
    (false negatives) carry higher operational risk than false alarms.
  • This model provides a solid baseline for an AI-powered cybersecurity
    monitoring system, and can be extended to real-time packet stream analysis.
''')

---
*UNSW-NB15 Dataset | AI-Based Network Intrusion Detection & Cybersecurity Analytics*